# ARES 2023-68A — Per-Loan LGD

Compute **LGD = 1 − S&P Recovery Rate** for each loan in the LLD tape and write the result to a derived CSV

**Units:** S&P Recovery Rate may be a fraction (e.g. 0.39) or a percent (e.g. 38.73). We normalize to a 0–1 fraction — divide by 100 only if values are > 1 — so LGD is always a 0–1 decimal.

In [5]:
import os, glob
import pandas as pd

# locate lld_ares_2023.xlsx regardless of launch dir
CANDIDATES = [
    'data/ARES_2023_68A', '../data/ARES_2023_68A',
    '../../data/ARES_2023_68A', '../../../data/ARES_2023_68A',
]
data_dir = next(p for p in CANDIDATES if os.path.isdir(p))
LLD_PATH = glob.glob(os.path.join(data_dir, '**', '*lld*.xlsx'), recursive=True)[0]
print('LLD:', LLD_PATH)

LLD: ../../data/ARES_2023_68A/lld_ares_2023.xlsx


In [6]:
# Real header is on the SECOND row (row index 1); row 0 is a title cell.
lld = pd.read_excel(LLD_PATH, header=1)
print("Initial Shape:", lld.shape)

# drop rows with a blank Issuer
lld = lld[lld['Issuer'].notna() & (lld['Issuer'].astype(str).str.strip() != '')].copy()
print('Shape after dropping rows with blank Issuer:', lld.shape)

Initial Shape: (415, 48)
Shape after dropping rows with blank Issuer: (415, 48)


In [7]:
# Normalize S&P Recovery Rate to a 0-1 fraction, then LGD = 1 - recovery.
rec = pd.to_numeric(lld['S&P Recovery Rate'], errors='coerce')
rec_frac = rec.where(rec <= 1, rec / 100.0)   # divide by 100 only if > 1
lld['LGD'] = 1.0 - rec_frac
print(lld[['Issuer', 'S&P Recovery Rate', 'LGD']].head(10).to_string(index=False))
print('\nLGD range:', round(lld['LGD'].min(), 4), '->', round(lld['LGD'].max(), 4))

                        Issuer  S&P Recovery Rate  LGD
Freeport LNG Investments, LLLP               0.40 0.60
                    Ensono, LP               0.35 0.65
        Tempo Acquisition, LLC               0.50 0.50
     Telenet Financing USD LLC               0.35 0.65
       SCIH Salt Holdings Inc.               0.28 0.72
              Proofpoint, Inc.               0.35 0.65
   Epicor Software Corporation               0.30 0.70
                RealPage, Inc.               0.35 0.65
                 Xplor T1, LLC               0.30 0.70
         Citadel Securities LP               0.50 0.50

LGD range: 0.25 -> 0.98


In [ ]:
# Write the loan tape with the new LGD column to a derived CSV.
OUT = 'output/lld_ares_2023_derived.csv'
lld.to_csv(OUT, index=False)
print('wrote', os.path.abspath(OUT), '| rows:', len(lld), '| LGD column added')

wrote /Users/amine/Documents/Columbia/Classes/ENGIE 4700 - Summer Project/correlation_and_tail_risk_in_clo_tranches/project/notebooks/output/lld_ares_2023_lgd.csv | rows: 415 | LGD column added
